<a href="https://colab.research.google.com/github/BahruzHuseynov/Portfolio/blob/main/Medium/PyTorch/PyTorch15_Transfer_Learning_and_Fine_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Importing Libraries

In [80]:
import torch
import torch.nn as nn
from torch import optim

from torchvision import transforms, datasets, models

from torch.utils.data import DataLoader, random_split

## Dataset and Data Loader

Dataset - https://www.kaggle.com/datasets/rogeriovaz/rome-weather-classification/data <br>
5 classes - 250 images <br>
50 images per class

In [81]:
num_classes = 5

In [82]:
transform = transforms.Compose([
    transforms.Resize((227, 227)),
    transforms.ToTensor(),
    transforms.Normalize([0.48, 0.45, 0.4], [0.3, 0.24, 0.225])
])

In [83]:
dataset = datasets.ImageFolder(root = "drive/MyDrive/Rome Weather", transform = transform)

In [84]:
print(dataset.classes)
print(dataset.class_to_idx)

['Cloudy', 'Foggy', 'Rainy', 'Snowy', 'Sunny']
{'Cloudy': 0, 'Foggy': 1, 'Rainy': 2, 'Snowy': 3, 'Sunny': 4}


### Train and Test split

In [85]:
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

In [86]:
train_loader = DataLoader(train_dataset, batch_size=10, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=5, shuffle=False)

## AlexNet

<img src = "https://neurohive.io/wp-content/uploads/2018/10/AlexNet-1.png">

In [87]:
# models.alexnet() - calling with random weights

alexnet = models.alexnet(pretrained=True) # calling with pretrained weights

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [88]:
alexnet

AlexNet(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 192, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (avgpool): AdaptiveAvgPool2d(output_size=(6, 6))
  (classifier): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=9216, out_features=4096, bias=True)
 

## Transfer Learning 1 - Training Only Fully Connected Layers

In this part, entire layers will be <b> frozen </b> and only last layer will be trained. <br>
We can apply this technique starting from any layer.

In [89]:
# Freezing all layers
for param in alexnet.parameters():
    param.requires_grad = False

In [90]:
last_layer_features = alexnet.classifier[6].in_features
alexnet.classifier[6] = nn.Linear(last_layer_features, num_classes)

### Training

In [91]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
alexnet.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(alexnet.classifier[6].parameters(), lr=0.001, momentum=0.9)

In [92]:
num_epochs = 10
for epoch in range(num_epochs):
    alexnet.train()
    train_loss = 0.0
    for batch_idx, (samples, targets) in enumerate(train_loader):
        samples = samples.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        outputs = alexnet(samples)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
    epoch_loss = train_loss / len(train_loader.dataset)
    print("Epoch:", epoch + 1, "--> Running loss:", train_loss, "--> Epoch loss:", epoch_loss)

Epoch: 1 --> Running loss: 23.28103744983673 --> Epoch loss: 0.11640518724918365
Epoch: 2 --> Running loss: 11.175675798207521 --> Epoch loss: 0.055878378991037606
Epoch: 3 --> Running loss: 5.48060142993927 --> Epoch loss: 0.02740300714969635
Epoch: 4 --> Running loss: 2.985753880464472 --> Epoch loss: 0.01492876940232236
Epoch: 5 --> Running loss: 2.5137800266966224 --> Epoch loss: 0.012568900133483113
Epoch: 6 --> Running loss: 2.261915510520339 --> Epoch loss: 0.011309577552601695
Epoch: 7 --> Running loss: 2.2319432450458407 --> Epoch loss: 0.011159716225229203
Epoch: 8 --> Running loss: 1.5179412942379713 --> Epoch loss: 0.007589706471189856
Epoch: 9 --> Running loss: 0.8289232000242919 --> Epoch loss: 0.0041446160001214595
Epoch: 10 --> Running loss: 1.468587446026504 --> Epoch loss: 0.007342937230132521


### Testing and Performance Checking

In [93]:
alexnet.eval()
preds = []
actual = []
test_loss = 0.0
with torch.no_grad():
    for samples, targets in test_loader:
        samples = samples.to(device)

        outputs = alexnet(samples)
        _, predicted = torch.max(outputs.data, 1)
        preds.extend(predicted.cpu().numpy())
        actual.extend(targets.numpy())

In [94]:
from sklearn.metrics import classification_report, accuracy_score

In [95]:
print(classification_report(actual, preds))

              precision    recall  f1-score   support

           0       0.67      0.80      0.73        10
           1       1.00      1.00      1.00         9
           2       0.75      0.50      0.60        12
           3       0.90      0.90      0.90        10
           4       0.73      0.89      0.80         9

    accuracy                           0.80        50
   macro avg       0.81      0.82      0.81        50
weighted avg       0.80      0.80      0.79        50



In [96]:
accuracy_score(actual, preds)

0.8

### Transfer Learning 2 - Fine Tuning (No freezing)

In [97]:
alexnet_mod = models.alexnet(pretrained=True)

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [106]:
last_layer_features = alexnet_mod.classifier[6].in_features
alexnet_mod.classifier[6] = nn.Linear(last_layer_features, num_classes)

alexnet_mod.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(alexnet_mod.parameters(), lr=0.001, momentum=0.9)

In [107]:
num_epochs = 8
for epoch in range(num_epochs):
    alexnet_mod.train()
    train_loss = 0.0
    for batch_idx, (samples, targets) in enumerate(train_loader):
        samples = samples.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        outputs = alexnet_mod(samples)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
    epoch_loss = train_loss / len(train_loader.dataset)
    print("Epoch:", epoch + 1, "--> Running loss:", train_loss, "--> Epoch loss:", epoch_loss)

Epoch: 1 --> Running loss: 12.350761622190475 --> Epoch loss: 0.06175380811095238
Epoch: 2 --> Running loss: 2.5152797093614936 --> Epoch loss: 0.012576398546807468
Epoch: 3 --> Running loss: 0.3584472187794745 --> Epoch loss: 0.0017922360938973725
Epoch: 4 --> Running loss: 0.613880468707066 --> Epoch loss: 0.00306940234353533
Epoch: 5 --> Running loss: 0.9152727865730412 --> Epoch loss: 0.004576363932865206
Epoch: 6 --> Running loss: 1.1203193221008405 --> Epoch loss: 0.005601596610504202
Epoch: 7 --> Running loss: 0.6455265623517334 --> Epoch loss: 0.003227632811758667
Epoch: 8 --> Running loss: 0.6648982289480045 --> Epoch loss: 0.0033244911447400226


In [108]:
alexnet_mod.eval()
preds = []
actual = []
test_loss = 0.0
with torch.no_grad():
    for samples, targets in test_loader:
        samples = samples.to(device)

        outputs = alexnet_mod(samples)
        _, predicted = torch.max(outputs.data, 1)
        preds.extend(predicted.cpu().numpy())
        actual.extend(targets.numpy())

In [109]:
print(classification_report(actual, preds))

              precision    recall  f1-score   support

           0       0.70      0.70      0.70        10
           1       0.90      1.00      0.95         9
           2       0.73      0.67      0.70        12
           3       0.82      0.90      0.86        10
           4       0.88      0.78      0.82         9

    accuracy                           0.80        50
   macro avg       0.80      0.81      0.80        50
weighted avg       0.80      0.80      0.80        50



In [110]:
accuracy_score(actual, preds)

0.8